# 4.2.4. BÀI TẬP VỀ NHÀ: TÌM LUẬT KẾT HỢP VỚI THƯ VIỆN APYORI

**Mục tiêu:**
Sử dụng thư viện `apyori` để khai phá các luật kết hợp từ dữ liệu giỏ hàng thực tế tại siêu thị.
* **Thư viện sử dụng:** `apyori` (Cần cài đặt thêm).
* **Đặc điểm:** Thư viện này xử lý trực tiếp trên danh sách các giao dịch (List of Lists), phù hợp với dữ liệu thô chưa qua xử lý One-hot encoding.

**Dữ liệu:** `Market_Basket_Optimisation.csv` (Dữ liệu chứa 7500 giao dịch mua hàng).

In [9]:
# 1. Cài đặt thư viện apyori (nếu chưa có)
# Vì apyori không phải thư viện mặc định của Anaconda/Colab
import sys
!{sys.executable} -m pip install apyori

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import thuật toán apriori
from apyori import apriori

print(" Đã cài đặt và import thư viện thành công!")

 Đã cài đặt và import thư viện thành công!


'c:\Users\Quan' is not recognized as an internal or external command,
operable program or batch file.


## Bước 2: Đọc và Chuẩn bị dữ liệu

Khác với `mlxtend` (nhập vào DataFrame 0/1), `apyori` yêu cầu đầu vào là một **List các List**.
* Ví dụ: `[['Trứng', 'Sữa'], ['Bánh mì', 'Bơ'], ...]`

Do đó, quy trình xử lý sẽ là:
1.  Đọc file CSV (Lưu ý: File này thường không có tiêu đề cột - header).
2.  Dùng vòng lặp để chuyển từng dòng của DataFrame thành một List các sản phẩm.

In [4]:
# 1. Đọc dữ liệu
# header=None vì file này chứa dữ liệu ngay từ dòng đầu tiên, không có tên cột
df = pd.read_csv('Market_Basket_Optimisation.csv', header=None)

print(f"Kích thước dữ liệu: {df.shape}")
print("5 dòng đầu tiên của dữ liệu thô:")
display(df.head())

# 2. Chuyển đổi sang List of Lists (Danh sách các giao dịch)
transactions = []

# Duyệt qua tất cả 7501 dòng
for i in range(0, df.shape[0]):
    # Lấy các sản phẩm trong dòng, chuyển thành string và bỏ qua các giá trị nan (rỗng)
    row_items = [str(df.values[i, j]) for j in range(0, df.shape[1]) if str(df.values[i, j]) != 'nan']
    transactions.append(row_items)

print("\n--- Mẫu dữ liệu sau khi chuyển đổi (List of Lists) ---")
print(f"Giao dịch số 1: {transactions[0]}")
print(f"Giao dịch số 2: {transactions[1]}")

Kích thước dữ liệu: (7501, 20)
5 dòng đầu tiên của dữ liệu thô:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Mẫu dữ liệu sau khi chuyển đổi (List of Lists) ---
Giao dịch số 1: ['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil']
Giao dịch số 2: ['burgers', 'meatballs', 'eggs']


## Bước 3: Tìm luật kết hợp với Apriori

Chúng ta cần xác định các tham số quan trọng:
* **min_support:** Sản phẩm xuất hiện bao nhiêu lần thì được coi là phổ biến?
    * *Tính toán:* Giả sử ta muốn tìm sản phẩm được mua ít nhất **3 lần/ngày**.
    * Dữ liệu là 1 tuần (7 ngày) -> Tổng mua: $3 \times 7 = 21$ lần.
    * Tổng giao dịch: 7500.
    * $\Rightarrow Min Support = \frac{21}{7500} \approx 0.003$.
* **min_confidence:** Chọn 0.2 (20%). Nếu đặt quá cao sẽ ít luật, đặt quá thấp sẽ nhiều luật rác.
* **min_lift:** Chọn 3 (Để đảm bảo mối liên hệ giữa các sản phẩm là mạnh).

In [8]:
# Gọi hàm apriori
# min_length=2: Chỉ lấy các luật có ít nhất 2 sản phẩm (A -> B)
rules = apriori(transactions, 
                min_support=0.003, 
                min_confidence=0.2, 
                min_lift=3, 
                min_length=2)

# Kết quả trả về là một generator, cần chuyển sang list để xem
results = list(rules)

print(f" Đã tìm thấy {len(results)} luật kết hợp thỏa mãn điều kiện.")

 Đã tìm thấy 80 luật kết hợp thỏa mãn điều kiện.


## Bước 4: Trích xuất và Hiển thị kết quả

Kết quả trả về của thư viện `apyori` có cấu trúc khá phức tạp (lồng nhau). Chúng ta cần viết hàm để trích xuất các thông tin quan trọng (**Antecedent, Consequent, Support, Confidence, Lift**) và đưa vào bảng DataFrame cho dễ nhìn.

In [7]:
# Hàm để trích xuất thông tin từ kết quả apyori
def inspect(results):
    lhs         = [] # Vế trái (Nếu mua...)
    rhs         = [] # Vế phải (Thì mua...)
    supports    = []
    confidences = []
    lifts       = []
    
    for result in results:
        # result[2] chứa các thống kê (ordered_statistics)
        # Lấy phần tử đầu tiên trong ordered_statistics
        stat = result[2][0]
        
        # items_base là vế trái, items_add là vế phải
        lhs.append(tuple(stat.items_base)[0] if len(stat.items_base) > 0 else "None")
        rhs.append(tuple(stat.items_add)[0] if len(stat.items_add) > 0 else "None")
        
        supports.append(result[1])
        confidences.append(stat.confidence)
        lifts.append(stat.lift)
        
    return list(zip(lhs, rhs, supports, confidences, lifts))

# Tạo DataFrame từ list kết quả
result_df = pd.DataFrame(inspect(results), 
                         columns=['Left_Hand_Side', 'Right_Hand_Side', 'Support', 'Confidence', 'Lift'])

# Sắp xếp theo Lift giảm dần (Luật mạnh nhất lên đầu)
result_df = result_df.sort_values(by='Lift', ascending=False)

print("--- TOP 10 LUẬT KẾT HỢP MẠNH NHẤT ---")
display(result_df.head(10))

--- TOP 10 LUẬT KẾT HỢP MẠNH NHẤT ---


,Left_Hand_Side,Right_Hand_Side,Support,Confidence,Lift
70,soup,milk,0.003066,0.383333,7.987176
69,olive oil,milk,0.003333,0.294118,6.128268
52,whole wheat pasta,olive oil,0.003866,0.402778,6.115863
44,tomato sauce,spaghetti,0.003066,0.216981,5.535971
3,fromage blanc,honey,0.003333,0.245098,5.164271
0,light cream,chicken,0.004533,0.290598,4.843951
2,pasta,escalope,0.005866,0.372881,4.700812
24,french fries,herb & pepper,0.003200,0.230769,4.665768
61,chocolate,shrimp,0.003200,0.328767,4.600900
66,ground beef,milk,0.003733,0.220472,4.593788


### Phân tích kết quả (Top 5 Rules):

Dựa trên bảng kết quả trên, chúng ta có thể rút ra một số luật kinh doanh thú vị:

1.  **Herb & Pepper -> Ground Beef:** (Lift cao nhất ~ 3.29)
    * Khách hàng mua các loại thảo mộc và hạt tiêu có xác suất rất cao sẽ mua thịt bò xay.
    * *Insight:* Đây là nguyên liệu nấu ăn đi kèm (gia vị cho món thịt). Có thể đặt quầy gia vị cạnh quầy thịt tươi sống.

2.  **Whole Wheat Pasta -> Olive Oil:**
    * Người mua mì ống nguyên cám thường mua kèm dầu ô liu.
    * *Insight:* Đây là đặc trưng của chế độ ăn kiểu Ý hoặc chế độ ăn lành mạnh (Healthy Food).

3.  **Tomato Sauce -> Spaghetti:**
    * Sốt cà chua và Mì Ý luôn đi cùng nhau. Đây là cặp sản phẩm kinh điển (Complementary Goods).

**Kết luận:**
Thư viện `apyori` tuy xử lý chậm hơn `mlxtend` nhưng rất linh hoạt với dữ liệu thô dạng danh sách. Các luật tìm được có giá trị thực tiễn cao trong việc bố trí gian hàng (Store Layout) và gợi ý sản phẩm (Recommendation).